# COMP-4002 — Machine Learning Research Project

This notebook provides a complete, end-to-end machine-learning workflow:

1. **Environment setup** — imports, reproducibility seed  
2. **Data loading** — scikit-learn built-in datasets (swap in your own CSV)  
3. **Exploratory Data Analysis (EDA)** — statistics, distributions, correlations  
4. **Preprocessing** — train/val/test split, feature scaling  
5. **Model training** — Logistic Regression, Random Forest, SVM, k-NN, Gradient Boosting  
6. **Evaluation** — accuracy, precision, recall, F1, AUC, confusion matrices  
7. **Model comparison** — side-by-side summary table  
8. **Feature importance** — top predictors from the best tree-based model  

---
> **How to use your own data:** in *Section 2* replace the `load_sklearn_dataset` call
> with `load_csv("../data/your_file.csv", target_column="label")` and adjust
> `CLASS_NAMES` accordingly.

## 1. Environment Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Add the project root to the path so the src/ helpers are importable
sys.path.insert(0, "..")
from src.data_loader import load_sklearn_dataset, load_csv
from src.preprocessing import split_data, scale_features
from src.evaluation import (
    compute_metrics,
    compare_models,
    plot_confusion_matrix,
    print_classification_report,
)

# Global settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

print(f"Python {sys.version}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## 2. Data Loading

Change `DATASET_NAME` to `"breast_cancer"`, `"wine"`, or `"digits"` to experiment
with different datasets without touching the rest of the notebook.

In [ ]:
DATASET_NAME = "iris"   # <-- change this to switch datasets

X, y, CLASS_NAMES = load_sklearn_dataset(DATASET_NAME)

print(f"Dataset : {DATASET_NAME}")
print(f"Shape   : {X.shape}")
print(f"Classes : {CLASS_NAMES}")
print(f"Target distribution:\n{y.value_counts().rename(index=dict(enumerate(CLASS_NAMES)))}")
X.head()

## 3. Exploratory Data Analysis

In [ ]:
print("=== Descriptive Statistics ===")
X.describe().T

In [ ]:
# Missing value check
missing = X.isnull().sum()
if missing.any():
    print("Columns with missing values:")
    print(missing[missing > 0])
else:
    print("No missing values detected.")

In [ ]:
# Feature distributions
n_cols = min(len(X.columns), 8)
cols = X.columns[:n_cols]
fig, axes = plt.subplots(2, (n_cols + 1) // 2, figsize=(14, 6))
axes = axes.flatten()
for i, col in enumerate(cols):
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        axes[i].hist(X.loc[y == cls_idx, col], alpha=0.6, label=cls_name, bins=20)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Feature Distributions by Class", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot (first 4 features to keep it readable)
plot_cols = list(X.columns[:4])
pair_df = X[plot_cols].copy()
pair_df["class"] = [CLASS_NAMES[i] for i in y]
sns.pairplot(pair_df, hue="class", diag_kind="kde", plot_kws={"alpha": 0.6})
plt.suptitle("Pairplot of First Four Features", y=1.02)
plt.show()

## 4. Preprocessing

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, test_size=0.2, val_size=0.1, random_state=RANDOM_STATE
)

X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)

print(f"Train : {X_train_s.shape}")
print(f"Val   : {X_val_s.shape}")
print(f"Test  : {X_test_s.shape}")

## 5. Model Training

Five classifiers are trained with sensible default hyperparameters.
Each model is also evaluated with 5-fold cross-validation on the *training* set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    ),
    "SVM (RBF)": SVC(
        kernel="rbf", probability=True, random_state=RANDOM_STATE
    ),
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    ),
}

cv_scores: dict[str, float] = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train_s, y_train, cv=5, scoring="accuracy"
    )
    cv_scores[name] = scores.mean()
    model.fit(X_train_s, y_train)
    print(f"{name:<25}  CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

## 6. Evaluation on the Validation Set

In [ ]:
val_results: dict[str, dict] = {}
for name, model in models.items():
    y_pred = model.predict(X_val_s)
    y_prob = model.predict_proba(X_val_s) if hasattr(model, "predict_proba") else None
    val_results[name] = compute_metrics(y_val, y_pred, y_prob)

val_df = compare_models(val_results)
print("=== Validation Set Results ===")
val_df.style.format("{:.4f}").highlight_max(color="lightgreen").highlight_min(color="#ffcccc")

In [ ]:
# Confusion matrices for all models on the validation set
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
if n_models == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_val_s)
    plot_confusion_matrix(y_val, y_pred, CLASS_NAMES, title=name, ax=ax)
plt.suptitle("Confusion Matrices — Validation Set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Final Evaluation on the Held-Out Test Set

Select the best model based on validation F1 and evaluate it once on the test set.

In [ ]:
best_model_name = val_df.index[0]
best_model = models[best_model_name]
print(f"Best model (by validation F1): {best_model_name}")

y_test_pred = best_model.predict(X_test_s)
y_test_prob = (
    best_model.predict_proba(X_test_s)
    if hasattr(best_model, "predict_proba")
    else None
)

test_metrics = compute_metrics(y_test, y_test_pred, y_test_prob)
print("\nTest Set Metrics:")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

In [ ]:
print_classification_report(y_test, y_test_pred, CLASS_NAMES)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(
    y_test, y_test_pred, CLASS_NAMES,
    title=f"{best_model_name} — Test Set Confusion Matrix",
    ax=ax,
)
plt.tight_layout()
plt.show()

## 8. Feature Importance

If the best model exposes `feature_importances_` (tree-based models), we visualize
the top features. For other models the coefficients / permutation importance could
be used instead — see the `sklearn` documentation.

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(
        best_model.feature_importances_, index=X_train.columns
    ).sort_values(ascending=False)

    plt.figure(figsize=(9, 4))
    sns.barplot(x=importances.values, y=importances.index, palette="viridis")
    plt.title(f"Feature Importances — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, "coef_"):
    coefs = pd.Series(
        np.abs(best_model.coef_).mean(axis=0), index=X_train.columns
    ).sort_values(ascending=False)

    plt.figure(figsize=(9, 4))
    sns.barplot(x=coefs.values, y=coefs.index, palette="viridis")
    plt.title(f"Mean |Coefficient| — {best_model_name}")
    plt.xlabel("|Coefficient|")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature importances directly.")

---
## Summary

| Step | Action |
|------|---------|
| Dataset | Loaded via `src/data_loader.py` |
| Split | 70 % train / 10 % validation / 20 % test |
| Scaling | `StandardScaler` fit on train set only |
| Models | Logistic Regression, Random Forest, SVM, k-NN, Gradient Boosting |
| Selection | Best validation F1 score |
| Final eval | Accuracy, Precision, Recall, F1, AUC on held-out test set |

**Next steps**
- Replace the built-in dataset with your own data in Section 2
- Add hyperparameter tuning (e.g. `GridSearchCV`) after Section 5
- Explore deep-learning models with PyTorch or TensorFlow